<a href="https://colab.research.google.com/github/fatahrahimi330/XST-Deepfake-Detection/blob/master/Prediction/Video_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# We use CNN + ViT + BiLSTM model to predict whether a video is deepfake or real.

In [ ]:
import torch
import cv2
import numpy as np
from PIL import Image
from torchvision import transforms
from facenet_pytorch import MTCNN
from CNN_ViT_BiLSTM import CNN_ViT_BiLSTM
import torch.nn as nn
import timm

## 1. Load the Trained Model

In [ ]:
class CNN_ViT_BiLSTM(nn.Module):
    def __init__(self, cnn_model='efficientnet_b0', vit_model='vit_base_patch16_224', lstm_hidden=256, lstm_layers=1):
        super(CNN_ViT_BiLSTM, self).__init__()

        # CNN backbone (pretrained)
        self.cnn = timm.create_model(cnn_model, pretrained=True)
        self.cnn.reset_classifier(0)  # remove classifier to get features
        cnn_feature_dim = self.cnn.num_features

        # Freeze CNN
        for param in self.cnn.parameters():
            param.requires_grad = False

        # ViT backbone (pretrained)
        self.vit = timm.create_model(vit_model, pretrained=True)
        self.vit.reset_classifier(0)
        vit_feature_dim = self.vit.num_features

         # Freeze ViT
        for param in self.vit.parameters():
            param.requires_grad = False

        # BiLSTM for temporal modeling
        self.lstm = nn.LSTM(
            input_size=cnn_feature_dim + vit_feature_dim, # Corrected input size
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True
        )


        # Fully connected layer for binary output
        self.fc = nn.Sequential(
            nn.Linear(lstm_hidden*2, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1)
        )

    def forward(self, x):
        """
        x: shape (B, T, C, H, W)
        B = batch size, T = number of frames per video
        """
        B, T, C, H, W = x.shape
        x_flat = x.view(B*T, C, H, W) # Flatten B and T dimensions for CNN and ViT processing

        # CNN feature extraction from original images
        cnn_raw_feat = self.cnn.forward_features(x_flat)
        # Apply global pooling and flatten for CNN
        cnn_feat = self.cnn.global_pool(cnn_raw_feat).flatten(1)

        # ViT feature extraction from original images (parallel processing)
        vit_raw_feat = self.vit.forward_features(x_flat)
        # Extract CLS token for ViT
        vit_feat = vit_raw_feat[:, 0]

        # Concatenate features
        combined_feat = torch.cat((cnn_feat, vit_feat), dim=1) # Combine features

        # Reshape to sequence for LSTM
        seq_feat = combined_feat.view(B, T, -1)

        # BiLSTM
        lstm_out, _ = self.lstm(seq_feat)  # lstm_out: (B, T, 2*hidden)
        lstm_last = lstm_out[:, -1, :]     # take last frame output

        # FC layer
        out = self.fc(lstm_last)
        return out

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = CNN_ViT_BiLSTM()  # same architecture as during training
model.load_state_dict(torch.load("/content/drive/MyDrive/copy_best_model.pth", map_location=device))
model.to(device)
model.eval()  # important for evaluation!

In [ ]:
# ── Classes (must match ImageFolder alphabetical order) ───────────────────────
classes = ["fake", "real"]  # fake=0, real=1

In [ ]:
# ── MTCNN (CPU only) ──────────────────────────────────────────────────────────
mtcnn = MTCNN(keep_all=False, device="cpu")

In [ ]:
# ── Exact same helper as training ─────────────────────────────────────────────
def expand_box(x1, y1, x2, y2, w, h, margin_ratio=0.25):
    bw = x2 - x1
    bh = y2 - y1
    mx = int(bw * margin_ratio)
    my = int(bh * margin_ratio)
    x1 = max(0, x1 - mx)
    y1 = max(0, y1 - my)
    x2 = min(w, x2 + mx)
    y2 = min(h, y2 + my)
    return x1, y1, x2, y2

## 2. Preprocess video into frames

In [ ]:
# ── Extract frames matching training preprocessing ────────────────────────────
def video_to_frames(video_path, num_frames=16, img_size=224, frame_skip=10, margin_ratio=0.25):
    cap = cv2.VideoCapture(video_path)
    frames = []
    frame_id = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_id % frame_skip == 0:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            boxes, _ = mtcnn.detect(rgb)

            if boxes is not None and len(boxes) > 0:
                x1, y1, x2, y2 = boxes[0]
                x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])

                h, w, _ = rgb.shape
                x1, y1, x2, y2 = expand_box(x1, y1, x2, y2, w, h, margin_ratio)

                face = rgb[y1:y2, x1:x2]
                if face.size > 0:
                    face = cv2.resize(face, (img_size, img_size))
                    frames.append(face)

        frame_id += 1

        if len(frames) == num_frames:
            break

    cap.release()

    if len(frames) == 0:
        raise ValueError("No faces detected in the video!")

    while len(frames) < num_frames:
        frames.append(frames[-1])

    return np.stack(frames)  # (T, H, W, C)

## 3. Apply the same transformations as training

In [ ]:
# ── Transforms (matches val_test_transform in training) ──────────────────────
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# ── Preprocess frames ─────────────────────────────────────────────────────────
def preprocess_frames(frames, transform):
    pil_frames = [Image.fromarray(f) for f in frames]
    frames_tensor = torch.stack([transform(f) for f in pil_frames])
    frames_tensor = frames_tensor.unsqueeze(0)   # (1, T, C, H, W)
    return frames_tensor

## 4. Predict

In [ ]:
# ── Predict ───────────────────────────────────────────────────────────────────
video_path = "01__outside_talking_still_laughing.mp4"

frames = video_to_frames(video_path, num_frames=16)
x = preprocess_frames(frames, val_transform).to(device)

with torch.no_grad():
    output = model(x)
    prob = torch.sigmoid(output).item()
    pred = int(prob > 0.5)


# fake=0, real=1 — matches ImageFolder alphabetical order
print(f"Probability of being Fake: {1 - prob:.4f}")
print(f"Probability of being Real: {prob:.4f}")
print(f"Prediction: {classes[pred]}")